In [ ]:
%load_ext autoreload
%autoreload 2
%config InlineBackend.figure_format = "retina"

In [ ]:
import networkx as nx
import numpy as np
from rich import print as rich_print

from qlinks.caging import (
    CageClassificationConfig,
    CageSearchConfig,
    CageSearcher,
    classify_cage_state,
)
from qlinks.basis.sectors import sector_mask_from_build_result
from qlinks.basis.configs import basis_configs_from_build_result
from qlinks.models import (
    SquareQLMModel,
    SquareQDMModel,
    TriangularQLMModel,
    TriangularQDMModel,
    HoneycombQLMModel,
    HoneycombQDMModel,
)
from qlinks.qec import (
    CodeSpace,
    CageSectorCollection,
    LocalErrorSet,
    diagnose_knill_laflamme,
    search_projected_logical_operators,
    diagnose_cage_code_candidate,
    diagnose_cage_result_code_candidates,
    diagnose_projected_error_algebra,
    diagnose_cage_collection_code_candidate,
    compute_cage_record_fingerprints,
    match_cage_records_across_sectors,
    diagnose_matched_cage_collection_code_candidates,
)

## Model definition

In [ ]:
model = SquareQDMModel(
    lx=4,
    ly=4,
    boundary_condition="periodic",
    coup_kin=-1.0,
    coup_pot=0.7,
)

In [ ]:
collection = CageSectorCollection.from_model_sectors(
    model,
    sector_labels=[(0, 0), (1, 0), (0, 1), (1, 1)],
    sector_fields=("winding_a", "winding_b"),
    signature=(0, 4),
    cage_search_config=CageSearchConfig(
        search_type="type1",
        # type1_kappas=(0,),
        # type2_kappas=(-2, 2),
        tolerance=1e-10,
        degenerate_basis_strategy="ipr",
        ipr_n_restarts=256,
        ipr_candidate_count=128,
        ipr_random_seed=1234,
        store_full_states=True,
    ),
    build_kwargs={
        "builder": "bitmask",
        "basis_solver": "dfs",
    },
    ambient_basis_mode="model",
)

collection.to_rich(max_entries=32)

In [ ]:
code = CodeSpace.from_cage_collection(
    collection,
    allow_rank_deficient=True,
)

# errors = LocalErrorSet.from_model(build_result=build_result)
errors = LocalErrorSet.from_layout(
    model.layout,
    max_weight=4,
    include_value_diagonal=True,
    include_projectors=True,
    include_transitions=True,
    max_errors=100,
)

report = diagnose_cage_collection_code_candidate(
    collection=collection,
    errors=errors,
    max_weight=1,
    include_error_algebra=True,
)

report.to_rich()

In [ ]:
match_report = match_cage_records_across_sectors(
    collection,
    errors,
    signature=(0, 6),
    sector_labels=[(0, 0), (1, 0), (0, 1), (1, 1)],
    records_per_sector=1,
    max_weight=1,
    fingerprint_mode="kl_diagonal",
    max_matches=10,
)

match_report.to_rich()

In [ ]:
scan = diagnose_matched_cage_collection_code_candidates(
    collection=collection,
    errors=errors,
    signature=(0, 6),
    sector_labels=[(0, 0), (1, 0), (0, 1), (1, 1)],
    records_per_sector=1,
    match_max_weight=1,
    diagnostic_max_weight=1,
    fingerprint_mode="kl_diagonal",
    max_matches=10,
    include_error_algebra=True,
)

print(scan.to_text())

In [ ]:
scan.to_summary_dict()